# BurnInWatch — Final Feature Audit & Correct Modeling Table v3

## Important correction

The previous audit found:
- 42 columns in the combined candidate CSV
- 39 unique header names
- 3 duplicated relationship columns
- 687 windows

However, the reconstruction step reported **0 relative features**, even though the earlier `burninwatch_features_relative_v2.csv` clearly contained the six relative-to-baseline features.

That means the v3 reconstruction did not carry those six features into the final modeling table.

This notebook fixes that by **merging feature-group CSVs on the actual metadata keys** rather than relying on row order.

Expected unique feature families:
- 21 absolute features
- 6 relative-change features
- 3 context-residual features
- 2 additional relationship features

Expected total: **32 unique model features**.

No model is trained here.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

PROCESSED = Path("processed")

META = [
    "device_id",
    "file_name",
    "file_role",
    "window_index",
    "window_size",
    "window_start_seconds",
    "window_end_seconds",
]

absolute = pd.read_csv(
    PROCESSED / "burninwatch_features_absolute_v2.csv"
)

relative = pd.read_csv(
    PROCESSED / "burninwatch_features_relative_v2.csv"
)

residual = pd.read_csv(
    PROCESSED / "burninwatch_features_residual_v2.csv"
)

relationship = pd.read_csv(
    PROCESSED / "burninwatch_features_relationship_v2.csv"
)

print("Absolute:", absolute.shape)
print("Relative:", relative.shape)
print("Residual:", residual.shape)
print("Relationship:", relationship.shape)

Absolute: (687, 28)
Relative: (687, 18)
Residual: (687, 15)
Relationship: (687, 12)


In [2]:
# ============================================================
# 1. VERIFY EACH GROUP HAS UNIQUE KEYS
# ============================================================

for name, df in {
    "absolute": absolute,
    "relative": relative,
    "residual": residual,
    "relationship": relationship,
}.items():
    missing_meta = [c for c in META if c not in df.columns]
    print(f"{name}: missing metadata columns -> {missing_meta}")

    duplicate_keys = df.duplicated(META).sum()
    print(f"{name}: duplicate metadata keys -> {duplicate_keys}")

    if duplicate_keys:
        display(
            df[df.duplicated(META, keep=False)]
            .sort_values(META)
            .head(20)
        )

absolute: missing metadata columns -> []
absolute: duplicate metadata keys -> 0
relative: missing metadata columns -> []
relative: duplicate metadata keys -> 0
residual: missing metadata columns -> []
residual: duplicate metadata keys -> 0
relationship: missing metadata columns -> []
relationship: duplicate metadata keys -> 0


## 2. Inspect the actual relative feature columns

This is the key diagnostic for the previous discrepancy.

In [4]:
# ============================================================
# 2. DISCOVER RELATIVE FEATURES ROBUSTLY
# ============================================================

print("Relative CSV columns:")
for i, c in enumerate(relative.columns):
    print(f"{i:02d}: {repr(c)}")

# Relative features have a very specific naming convention.
relative_features = [
    c for c in relative.columns
    if c.endswith("_delta_from_baseline")
    or c.endswith("_pct_change_from_baseline")
]

print("\nRelative feature columns found:")
for c in relative_features:
    print(" -", c)

print("\nCount:", len(relative_features))

Relative CSV columns:
00: 'device_id'
01: 'file_name'
02: 'file_role'
03: 'window_index'
04: 'window_size'
05: 'window_start_seconds'
06: 'window_end_seconds'
07: 'node2Voltage_delta_from_baseline'
08: 'node2Voltage_pct_change_from_baseline'
09: 'collectorEmitterCurrent_delta_from_baseline'
10: 'collectorEmitterCurrent_pct_change_from_baseline'
11: 'packageTemperature_delta_from_baseline'
12: 'packageTemperature_pct_change_from_baseline'
13: 'node2Voltage_baseline'
14: 'collectorEmitterCurrent_baseline'
15: 'packageTemperature_baseline'
16: 'expected_current'
17: 'expected_package_temperature'

Relative feature columns found:
 - node2Voltage_delta_from_baseline
 - node2Voltage_pct_change_from_baseline
 - collectorEmitterCurrent_delta_from_baseline
 - collectorEmitterCurrent_pct_change_from_baseline
 - packageTemperature_delta_from_baseline
 - packageTemperature_pct_change_from_baseline

Count: 6


## 3. Define non-duplicated feature groups

The three relationship variables below already occur in the absolute dataset, so they are not added twice:

- `supply_node1_mean_diff`
- `node1_node2_mean_diff`
- `estimated_power_proxy`

Only the two genuinely new relationship features are added from the relationship file.

In [ ]:
# ============================================================
# 2B. VERIFY RELATIVE FEATURES
# ============================================================

expected_relative_features = {
    "node2Voltage_delta_from_baseline",
    "node2Voltage_pct_change_from_baseline",
    "collectorEmitterCurrent_delta_from_baseline",
    "collectorEmitterCurrent_pct_change_from_baseline",
    "packageTemperature_delta_from_baseline",
    "packageTemperature_pct_change_from_baseline",
}

found_relative_features = set(relative_features)

missing = expected_relative_features - found_relative_features
unexpected = found_relative_features - expected_relative_features

print("Expected:", len(expected_relative_features))
print("Found:", len(found_relative_features))

print("\nMissing:", missing)
print("Unexpected:", unexpected)

assert not missing, f"Missing relative features: {missing}"
assert len(relative_features) == 6

print("\n✅ All 6 relative features found.")

## 4. Merge all groups using stable metadata keys

We use explicit key-based left joins.

This prevents accidental feature misalignment if one source file is sorted differently from another.

In [ ]:
clean = absolute[META + absolute_features].copy()

for df, cols, label in [
    (
        relative,
        relative_features,
        "relative"
    ),
    (
        residual,
        residual_features,
        "residual"
    ),
    (
        relationship,
        relationship_features,
        "relationship"
    ),
]:
    merge_df = df[META + cols].copy()

    clean = clean.merge(
        merge_df,
        on=META,
        how="left",
        validate="one_to_one"
    )

    print(f"After merging {label}: {clean.shape}")

In [ ]:
# ============================================================
# 5. FINAL MODEL FEATURE LIST
# ============================================================

MODEL_FEATURES = (
    absolute_features
    + relative_features
    + residual_features
    + relationship_features
)

print("Total model features:", len(MODEL_FEATURES))

# Verify uniqueness.
assert len(MODEL_FEATURES) == len(set(MODEL_FEATURES))

print("\nDuplicate feature names:")
print(
    pd.Series(MODEL_FEATURES)[
        pd.Series(MODEL_FEATURES).duplicated()
    ].tolist()
)

print("\nModel features:")
for i, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{i:02d}. {feature}")

## 6. Audit the corrected model table

We check missingness, finite values, variation, and numerical range.

In [ ]:
audit_rows = []

for feature in MODEL_FEATURES:
    s = pd.to_numeric(clean[feature], errors="coerce")

    audit_rows.append({
        "feature": feature,
        "missing_fraction": s.isna().mean(),
        "finite_fraction": np.isfinite(s).mean(),
        "nunique": s.nunique(dropna=True),
        "std": s.std(),
        "min": s.min(),
        "max": s.max(),
    })

audit = pd.DataFrame(audit_rows)

display(audit)

In [ ]:
# ============================================================
# 7. SELECT USABLE FEATURES
# ============================================================

usable_features = [
    row["feature"]
    for _, row in audit.iterrows()
    if (
        row["finite_fraction"] >= 0.99
        and row["nunique"] > 1
        and np.isfinite(row["std"])
        and row["std"] > 0
    )
]

print("All model features:", len(MODEL_FEATURES))
print("Usable features:", len(usable_features))

removed = [
    c for c in MODEL_FEATURES
    if c not in usable_features
]

print("Removed:", removed)

## 8. Check redundancy after the reconstruction

We do not remove correlated features automatically. We first record the strongest correlations for documented feature selection.

In [ ]:
corr = clean[usable_features].corr()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2",
        0: "correlation"
    })
)

high_corr = (
    high_corr[
        high_corr["correlation"].abs() >= 0.95
    ]
    .sort_values(
        "correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
)

display(high_corr.head(50))

## 9. Create the final ML-ready table

Metadata remains in the CSV for:
- device-aware splitting
- experiment traceability
- visualization
- later interpretation

But `device_id`, `file_name`, `file_role`, and window identifiers will **never** be passed as numerical model inputs.

In [ ]:
modeling_columns = [
    "sample_id",
] + META + usable_features

clean.insert(
    0,
    "sample_id",
    np.arange(len(clean))
)

modeling = clean[modeling_columns].copy()

path = PROCESSED / "burninwatch_modeling_dataset_v3_fixed.csv"

modeling.to_csv(path, index=False)

audit.to_csv(
    PROCESSED / "burninwatch_feature_audit_v3_fixed.csv",
    index=False
)

print("Saved:", path)
print("Shape:", modeling.shape)
print("Duplicate columns:", modeling.columns[modeling.columns.duplicated()].tolist())

In [ ]:
# ============================================================
# 10. FINAL SANITY CHECK
# ============================================================

print("Rows:", len(modeling))
print("Metadata columns:", META)
print("Model feature count:", len(usable_features))

print("\nRelative features present in final table:")
print([
    c for c in relative_features
    if c in modeling.columns
])

assert len(relative_features) == 6
assert all(c in modeling.columns for c in relative_features)
assert len(modeling.columns) == 1 + len(META) + len(usable_features)

print("\nSUCCESS: corrected v3 modeling table is internally consistent.")

# STOP POINT

Do not train yet.

We now want the corrected output to confirm:

- 687 rows
- 32 unique model-feature candidates before usability filtering
- all 6 relative-to-baseline features present
- no duplicate feature columns

After that, the next notebook will build the **device-aware unsupervised train/validation/test protocol**.

Because only four physical devices are represented, split design is more important than simply calling `train_test_split()`.